# GEDI Cal/Val MAAP STAC OGCification Demo

This notebook demonstrates the full AI-assisted OGC/MAAP DPS packaging workflow using a dataset already available in the MAAP STAC stack:

- STAC Browser collection: https://stac-browser.maap-project.org/collections/GEDI_CalVal_Lidar_Data
- STAC API collection: https://stac.maap-project.org/collections/GEDI_CalVal_Lidar_Data

The notebook is intentionally written like a MAAP documentation tutorial. Some cells are exploratory/notebook-only, and some cells are structured to become a DPS/OGC job. The generated reports should show which parts are safe to package and which parts should stay in the notebook.


## 1. Demo goals

We will:

1. Read real collection/item metadata from MAAP STAC.
2. Recommend an optimized algorithm from a provided algorithm catalog.
3. Run a small DPS-ready metadata summarization job.
4. Write outputs under `output/`.
5. Keep optional mapping/visualization notebook-only.
6. Use the repository generator to produce OGC/CWL/MAAP DPS package files and a suggested notebook V2.

This demo does **not** download full LAS point clouds by default. That is intentional: a metadata-first job is much faster and safer for a packaging demonstration. A full LAS-processing algorithm is listed as an alternative candidate, but it requires a larger runtime environment and staged input handling.


In [ ]:
# Papermill-style parameters. The generator turns these into runtime inputs.
stac_api_url = "https://stac.maap-project.org"
stac_browser_collection_url = "https://stac-browser.maap-project.org/collections/GEDI_CalVal_Lidar_Data"
collection_id = "GEDI_CalVal_Lidar_Data"
collection_concept_id = ""
bbox = ""  # Optional: min_lon,min_lat,max_lon,max_lat
datetime = ""  # Optional STAC datetime filter
limit = 5
asset_key = "data"
access_mode = "stac_metadata"
preferred_algorithm = "auto"
output_dir = "output"


## 2. Setup

This setup cell is required by the DPS-ready cells. The suggested notebook V2 generator should preserve setup/import cells like this so converted runtime code does not lose dependencies.


In [ ]:
import csv
import json
from pathlib import Path
from statistics import mean
from urllib.parse import urlencode

import requests


## 3. Algorithm catalog and recommendation

A real AI-assisted system can choose from a set of known algorithm patterns. This cell provides a small catalog for the demo. The recommended algorithm is metadata-first by default because it avoids downloading large LAS point clouds while still proving the STAC discovery, package generation, output writing, and validation flow.


In [ ]:
ALGORITHM_CATALOG = [
    {
        "name": "stac_metadata_inventory",
        "description": "Summarize STAC item metadata and LAS asset references without downloading point clouds.",
        "dps_suitability": "high",
        "estimated_runtime": "small",
        "requires_staged_input": False,
        "recommended_for_demo": True,
    },
    {
        "name": "las_asset_manifest_for_staging",
        "description": "Create a manifest of LAS S3 assets for a downstream staged-input DPS job.",
        "dps_suitability": "high",
        "estimated_runtime": "small",
        "requires_staged_input": False,
        "recommended_for_demo": True,
    },
    {
        "name": "full_las_pointcloud_metrics",
        "description": "Download or stage LAS files and compute point-cloud height metrics.",
        "dps_suitability": "medium",
        "estimated_runtime": "large",
        "requires_staged_input": True,
        "recommended_for_demo": False,
    },
    {
        "name": "canopy_height_rasterization",
        "description": "Rasterize LAS point clouds into canopy-height products.",
        "dps_suitability": "medium",
        "estimated_runtime": "large",
        "requires_staged_input": True,
        "recommended_for_demo": False,
    },
]


def recommend_algorithm(catalog, preferred_algorithm="auto"):
    if preferred_algorithm and preferred_algorithm != "auto":
        matches = [item for item in catalog if item["name"] == preferred_algorithm]
        if matches:
            return matches[0], catalog

    ranked = sorted(
        catalog,
        key=lambda item: (
            item["recommended_for_demo"],
            item["dps_suitability"] == "high",
            item["estimated_runtime"] == "small",
            not item["requires_staged_input"],
        ),
        reverse=True,
    )
    return ranked[0], ranked


selected_algorithm, ranked_algorithms = recommend_algorithm(ALGORITHM_CATALOG, preferred_algorithm)
selected_algorithm


## 4. DPS-ready STAC discovery helpers

These functions are suitable for a DPS job because they take explicit parameters, avoid hidden notebook state, and return serializable dictionaries/lists.


In [ ]:
def parse_bbox(bbox_text):
    if not bbox_text:
        return None
    values = [float(part.strip()) for part in bbox_text.split(",") if part.strip()]
    if len(values) != 4:
        raise ValueError("bbox must be min_lon,min_lat,max_lon,max_lat")
    return values


def stac_get_json(url, params=None, timeout=30):
    response = requests.get(url, params=params or {}, timeout=timeout)
    response.raise_for_status()
    return response.json()


def fetch_collection_metadata(stac_api_url, collection_id):
    url = f"{stac_api_url.rstrip('/')}/collections/{collection_id}"
    return stac_get_json(url)


def fetch_collection_items(stac_api_url, collection_id, limit=5, bbox="", datetime=""):
    url = f"{stac_api_url.rstrip('/')}/collections/{collection_id}/items"
    params = {"limit": int(limit)}
    parsed_bbox = parse_bbox(bbox)
    if parsed_bbox:
        params["bbox"] = ",".join(str(value) for value in parsed_bbox)
    if datetime:
        params["datetime"] = datetime
    return stac_get_json(url, params=params)


## 5. DPS-ready processing helpers

The selected algorithm summarizes STAC item metadata and asset references. It does not download large LAS files. This is the recommended demo path for fast OGC/DPS packaging.


In [ ]:
def asset_href_for_item(item, asset_key="data"):
    assets = item.get("assets") or {}
    asset = assets.get(asset_key) or next(iter(assets.values()), {})
    return asset.get("href", "") if isinstance(asset, dict) else ""


def summarize_items(items_response, asset_key="data"):
    features = items_response.get("features", [])
    summaries = []
    sizes_mb = []
    for feature in features:
        properties = feature.get("properties", {})
        size_value = properties.get("granule_size")
        try:
            sizes_mb.append(float(size_value))
        except (TypeError, ValueError):
            pass
        summaries.append(
            {
                "id": feature.get("id", ""),
                "datetime": properties.get("datetime", ""),
                "concept_id": properties.get("concept_id", ""),
                "collection_concept_id": properties.get("collection_concept_id", ""),
                "producer_granule_id": properties.get("producer_granule_id", ""),
                "bbox": feature.get("bbox", []),
                "asset_href": asset_href_for_item(feature, asset_key),
            }
        )

    return {
        "item_count": len(summaries),
        "asset_count": sum(1 for item in summaries if item["asset_href"]),
        "mean_granule_size_mb": mean(sizes_mb) if sizes_mb else None,
        "items": summaries,
    }


def build_job_result(collection_metadata, items_response, selected_algorithm, ranked_algorithms, asset_key="data"):
    item_summary = summarize_items(items_response, asset_key=asset_key)
    return {
        "collection_id": collection_metadata.get("id", collection_id),
        "collection_title": collection_metadata.get("title", ""),
        "collection_description": collection_metadata.get("description", "")[:500],
        "selected_algorithm": selected_algorithm,
        "algorithm_recommendations": ranked_algorithms,
        "stac_browser_collection_url": stac_browser_collection_url,
        "item_summary": item_summary,
    }


## 6. Run DPS-ready discovery and processing

This is the main job-like section. It fetches collection/items metadata and computes a compact summary. If the STAC service is temporarily unavailable, it writes an explicit error object instead of relying on hidden notebook state.


In [ ]:
try:
    collection_metadata = fetch_collection_metadata(stac_api_url, collection_id)
    items_response = fetch_collection_items(stac_api_url, collection_id, limit=limit, bbox=bbox, datetime=datetime)
    job_result = build_job_result(collection_metadata, items_response, selected_algorithm, ranked_algorithms, asset_key)
except Exception as exc:
    job_result = {
        "collection_id": collection_id,
        "selected_algorithm": selected_algorithm,
        "algorithm_recommendations": ranked_algorithms,
        "error": str(exc),
        "item_summary": {"item_count": 0, "asset_count": 0, "items": []},
    }

job_result["item_summary"]["item_count"]


## 7. DPS-ready output writing

DPS/OGC jobs must write products under `output/` or the configured `output_dir`. The generated package validator checks this pattern.


In [ ]:
def write_job_outputs(job_result, output_dir="output"):
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    summary_path = output_path / "gedi_calval_summary.json"
    algorithm_path = output_path / "algorithm_recommendations.json"
    manifest_path = output_path / "asset_manifest.csv"

    summary_path.write_text(json.dumps(job_result, indent=2) + "\n", encoding="utf-8")
    algorithm_path.write_text(json.dumps(job_result.get("algorithm_recommendations", []), indent=2) + "\n", encoding="utf-8")

    with manifest_path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=["id", "datetime", "concept_id", "asset_href"])
        writer.writeheader()
        for item in job_result.get("item_summary", {}).get("items", []):
            writer.writerow(
                {
                    "id": item.get("id", ""),
                    "datetime": item.get("datetime", ""),
                    "concept_id": item.get("concept_id", ""),
                    "asset_href": item.get("asset_href", ""),
                }
            )

    return {
        "summary": str(summary_path),
        "algorithm_recommendations": str(algorithm_path),
        "asset_manifest": str(manifest_path),
    }


written_outputs = write_job_outputs(job_result, output_dir)
written_outputs


## 8. Optional notebook-only map preview

This section is useful for a human demo but should not be required for DPS execution. It uses `folium` when available and reads only the metadata already produced above.


In [ ]:
try:
    import folium

    first_item = job_result.get("item_summary", {}).get("items", [{}])[0]
    first_bbox = first_item.get("bbox") or [-110.98, 31.72, -110.74, 31.92]
    center_lat = (first_bbox[1] + first_bbox[3]) / 2
    center_lon = (first_bbox[0] + first_bbox[2]) / 2
    preview_map = folium.Map(location=[center_lat, center_lon], zoom_start=9)
    folium.Rectangle(
        bounds=[[first_bbox[1], first_bbox[0]], [first_bbox[3], first_bbox[2]]],
        tooltip="First GEDI Cal/Val STAC item bbox",
    ).add_to(preview_map)
    preview_map
except Exception as exc:
    print(f"Optional map preview skipped: {exc}")


## 9. Demo commands

From the repository root, run:

```bash
python3 generator/generate_package.py \
  --input notebooks/gedi_calval_maap_stack_ogcification_demo.ipynb \
  --scan-dps-readiness \
  --use-mcp-defaults \
  --llm-recommendations \
  --build-dependency-graph \
  --emit-suggested-notebook-v2 \
  --validate-ogc \
  --output-dir generated_gedi_calval_demo
```

Then validate:

```bash
cd generated_gedi_calval_demo
cwltool --validate application.cwl
cwltool --validate workflow.cwl
ap-validator --format json --detail all application-package.cwl
./run.sh
```
